# LAB 2: Variational Autoencoder (VAE) for Image Generation
**Datasets:** MNIST and Fashion-MNIST  
**Framework:** TensorFlow / Keras

## Aim
To implement a Variational Autoencoder, train it on MNIST and Fashion-MNIST, reconstruct images, and generate new samples from random latent vectors.

## VAE architecture
Input image (28×28×1) → Encoder (convolution + pooling) → latent distribution parameters (`z_mean`, `z_log_var`) → reparameterization sampling → Decoder (upsampling + convolution) → reconstructed image (28×28×1).

The objective is the sum of:
- **Reconstruction loss:** binary cross-entropy between input and reconstructed pixels.
- **KL-divergence:** regularizes the learned latent distribution toward a standard normal distribution.

## Pseudocode
1. Load MNIST and Fashion-MNIST.
2. Normalize image pixels to [0, 1].
3. Build an encoder that predicts `z_mean` and `z_log_var`.
4. Sample latent vectors using the reparameterization trick.
5. Build a decoder that maps latent vectors to images.
6. Train using reconstruction loss + KL-divergence.
7. Plot losses, reconstruct test images, and generate images from random latent vectors.

## Cell 1 — Install and import libraries

In [ ]:
%pip install -q tensorflow matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))

## Cell 2 — Load MNIST and Fashion-MNIST

In [ ]:
(x_train_mnist, _), (x_test_mnist, _) = tf.keras.datasets.mnist.load_data()
(x_train_fashion, _), (x_test_fashion, _) = tf.keras.datasets.fashion_mnist.load_data()

print("MNIST:", x_train_mnist.shape, x_test_mnist.shape)
print("Fashion-MNIST:", x_train_fashion.shape, x_test_fashion.shape)

## Cell 3 — View sample images

In [ ]:
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_train_mnist[i], cmap="gray")
    plt.axis("off")
    plt.subplot(2, 10, i + 11)
    plt.imshow(x_train_fashion[i], cmap="gray")
    plt.axis("off")
plt.figtext(0.01, 0.73, "MNIST", fontsize=11)
plt.figtext(0.01, 0.28, "Fashion-MNIST", fontsize=11)
plt.suptitle("Sample images: MNIST (top), Fashion-MNIST (bottom)")
plt.tight_layout()
plt.show()

## Cell 4 — Preprocess images

In [ ]:
def preprocess_images(images):
    images = images.astype("float32") / 255.0
    return np.expand_dims(images, axis=-1)

x_train_mnist = preprocess_images(x_train_mnist)
x_test_mnist = preprocess_images(x_test_mnist)
x_train_fashion = preprocess_images(x_train_fashion)
x_test_fashion = preprocess_images(x_test_fashion)

print("MNIST:", x_train_mnist.shape, "pixel range:", x_train_mnist.min(), x_train_mnist.max())
print("Fashion-MNIST:", x_train_fashion.shape, "pixel range:", x_train_fashion.min(), x_train_fashion.max())

## Cell 5 — Define the sampling layer

The reparameterization trick samples
\(z = \mu + \exp(0.5\log\sigma^2)\epsilon\), where \(\epsilon\sim\mathcal{N}(0,I)\).
This allows gradients to pass through the sampling operation during training.

In [ ]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

## Cell 6 — Build the encoder and decoder

In [ ]:
LATENT_DIM = 16

def build_encoder(latent_dim=LATENT_DIM):
    encoder_inputs = layers.Input(shape=(28, 28, 1), name="encoder_input")

    x = layers.Conv2D(32, 3, activation="relu", padding="same")(encoder_inputs)
    x = layers.MaxPooling2D(2, padding="same")(x)       # 14 × 14 × 32
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, padding="same")(x)       # 7 × 7 × 64
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)

    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling(name="z_sampling")([z_mean, z_log_var])

    return models.Model(
        encoder_inputs, [z_mean, z_log_var, z], name="encoder"
    )


def build_decoder(latent_dim=LATENT_DIM):
    latent_inputs = layers.Input(shape=(latent_dim,), name="latent_vector")

    x = layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)
    x = layers.Reshape((7, 7, 64))(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.UpSampling2D(2)(x)                       # 14 × 14 × 64
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.UpSampling2D(2)(x)                       # 28 × 28 × 32
    decoder_outputs = layers.Conv2D(
        1, 3, activation="sigmoid", padding="same", name="decoded_image"
    )(x)

    return models.Model(latent_inputs, decoder_outputs, name="decoder")


encoder = build_encoder()
decoder = build_decoder()

print("ENCODER ARCHITECTURE")
encoder.summary()
print("\nDECODER ARCHITECTURE")
decoder.summary()

## Cell 7 — Create the VAE with reconstruction and KL losses

In [ ]:
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, inputs, training=False):
        z_mean, z_log_var, z = self.encoder(inputs, training=training)
        return self.decoder(z, training=training)

    def _losses_for_batch(self, data, training):
        if isinstance(data, (tuple, list)):
            x = data[0]
        else:
            x = data

        z_mean, z_log_var, z = self.encoder(x, training=training)
        reconstruction = self.decoder(z, training=training)

        # BCE is reduced over image channels by Keras; sum over spatial dimensions.
        bce = tf.keras.losses.binary_crossentropy(x, reconstruction)
        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(bce, axis=(1, 2))
        )

        kl_per_image = -0.5 * tf.reduce_sum(
            1.0 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
            axis=1
        )
        kl_loss = tf.reduce_mean(kl_per_image)

        total_loss = reconstruction_loss + kl_loss
        return x, total_loss, reconstruction_loss, kl_loss

    def train_step(self, data):
        with tf.GradientTape() as tape:
            _, total_loss, reconstruction_loss, kl_loss = self._losses_for_batch(data, True)

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {metric.name: metric.result() for metric in self.metrics}

    def test_step(self, data):
        _, total_loss, reconstruction_loss, kl_loss = self._losses_for_batch(data, False)

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {metric.name: metric.result() for metric in self.metrics}


def build_vae(latent_dim=LATENT_DIM):
    enc = build_encoder(latent_dim)
    dec = build_decoder(latent_dim)
    vae = VAE(enc, dec, name="variational_autoencoder")
    vae.compile(optimizer=tf.keras.optimizers.Adam())
    return vae, enc, dec


vae, encoder, decoder = build_vae()
print("VAE created successfully.")

## Cell 8 — Train the VAE

To keep the first run manageable, the default is 5 epochs. Increase `EPOCHS` to 10–20 for a longer practical run. Train on one dataset at a time.

In [ ]:
EPOCHS = 5
BATCH_SIZE = 128

# Select which dataset to train on: "mnist" or "fashion"
DATASET_NAME = "mnist"

if DATASET_NAME.lower() == "mnist":
    x_train, x_test = x_train_mnist, x_test_mnist
elif DATASET_NAME.lower() in ("fashion", "fashion-mnist", "fashion_mnist"):
    x_train, x_test = x_train_fashion, x_test_fashion
else:
    raise ValueError("DATASET_NAME must be 'mnist' or 'fashion'.")

history = vae.fit(
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test, x_test),
    verbose=1
)

## Cell 9 — Plot total, reconstruction, and KL losses

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training total loss")
plt.plot(history.history["val_loss"], label="Validation total loss")
plt.title(f"VAE Loss — {DATASET_NAME}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history.history["reconstruction_loss"], label="Reconstruction loss")
plt.plot(history.history["kl_loss"], label="KL-divergence loss")
plt.title(f"Training Loss Components — {DATASET_NAME}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Cell 10 — Compare original and reconstructed images

In [ ]:
reconstructed = vae.predict(x_test[:10], verbose=0)

plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Original")

    plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Reconstructed")

plt.suptitle(f"Original vs Reconstructed — {DATASET_NAME}")
plt.tight_layout()
plt.show()

## Cell 11 — Generate new images from random latent vectors

In [ ]:
num_samples = 20
random_latent_vectors = tf.random.normal(shape=(num_samples, LATENT_DIM))
generated_images = decoder.predict(random_latent_vectors, verbose=0)

plt.figure(figsize=(10, 5))
for i in range(num_samples):
    plt.subplot(4, 5, i + 1)
    plt.imshow(generated_images[i].squeeze(), cmap="gray")
    plt.axis("off")

plt.suptitle(f"New Samples Generated from Random Latent Vectors — {DATASET_NAME}")
plt.tight_layout()
plt.show()

## Cell 12 — Evaluate and save the model

In [ ]:
# Evaluate with the VAE's custom test_step
evaluation = vae.evaluate(x_test, batch_size=BATCH_SIZE, return_dict=True, verbose=1)
print("Test metrics:", evaluation)

vae.save_weights(f"vae_{DATASET_NAME}_weights.weights.h5")
encoder.save(f"vae_{DATASET_NAME}_encoder.keras")
decoder.save(f"vae_{DATASET_NAME}_decoder.keras")

print("Encoder and decoder saved. VAE weights saved.")

## Cell 13 — Optional: train on the other dataset

Run this cell if you want to train on both MNIST and Fashion-MNIST. It creates a fresh model so the two experiments remain separate.

```python
# Choose the dataset not used above:
# If DATASET_NAME == "mnist", this trains Fashion-MNIST; otherwise it trains MNIST.

if DATASET_NAME.lower() == "mnist":
    second_name = "fashion"
    second_train, second_test = x_train_fashion, x_test_fashion
else:
    second_name = "mnist"
    second_train, second_test = x_train_mnist, x_test_mnist

vae2, encoder2, decoder2 = build_vae()
history2 = vae2.fit(
    second_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(second_test,),
    verbose=1
)

second_generated = decoder2.predict(
    tf.random.normal((20, LATENT_DIM)), verbose=0
)

plt.figure(figsize=(10, 4))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(second_generated[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle(f"Generated samples — {second_name}")
plt.tight_layout()
plt.show()
```

## Result and conclusion
A convolutional Variational Autoencoder was designed and trained on MNIST or Fashion-MNIST. The encoder learns the mean and log-variance of a latent distribution, the sampling layer applies the reparameterization trick, and the decoder reconstructs images. Reconstruction loss and KL-divergence jointly train the model. Random latent vectors are decoded to generate new samples.

## Viva questions
1. **What is a VAE?** A generative model that learns a probability distribution in latent space and samples from it to generate data.
2. **How is a VAE different from a conventional autoencoder?** A conventional autoencoder typically maps inputs to deterministic latent codes; a VAE learns a distribution characterized by a mean and log-variance and regularizes it using KL-divergence.
3. **What is latent space?** The lower-dimensional representation where the model encodes important image features.
4. **What is KL-divergence used for?** It encourages the learned latent distribution to remain close to a standard normal distribution.
5. **Why can a VAE generate new samples?** Its latent space is regularized so that sampling latent vectors from a standard normal distribution can produce plausible decoder outputs.